In [3]:
import subprocess, shutil, copy
from TprReader import TprReader

In [6]:
def Pressure(fname):
    reader = TprReader(fname)
    ref_p = [
        100, 0, 0,
        0, 100, 0,
        0, 0, 100
    ]
    compress = [
        4.5E-5, 0, 0,
        0, 4.5E-5, 0,
        0, 0, 4.5E-5
    ]
    assert len(ref_p) == 9
    assert len(compress) == 9
    reader.set_pressure('No', 'Isotropic', 1.0, ref_p, compress)


In [8]:
def run_cmd(cmd:str):
    ret = subprocess.run(cmd, shell=True)
    if ret.returncode != 0:
        raise Exception('\nError occurred from command: \n\t%s!!!' %cmd)
    
def MD(inittpr:str, nsteps:int = 10):
    reader = TprReader(inittpr)
    coords = reader.get_xvf('X') # get coords from tpr
    natmA = 120
    natmB = 132
    natm = natmA+natmB

    assert natm == coords.shape[0]
    for i in range(nsteps):
        # move two molecules distance of Z axis each 2.0 A
        tempcoords = copy.deepcopy(coords)
        tempcoords[:natmA,     2] += 0.05 * i
        tempcoords[natmA:natm, 2] -= 0.05 * i
        reader.set_xvf('X', tempcoords)
        # rename new.tpr to em_{i}.tpr
        suffix = inittpr.split(".tpr")[0]+"_"+str(i)
        shutil.move("new.tpr", f"{suffix}.tpr")
        run_cmd(f'gmx mdrun -deffnm {suffix} -v')
    print("Finished!")

In [4]:
def Temperature(fname:str, nsteps:int=2):
    tau_t = [1.0, 1.0] # coupling constant
    reader = TprReader(fname)
    for i in range(nsteps):
        ref_t = [100+i*100, 100+i*100] # 100, 200, 300 K
        reader.set_temperature("Vrescale", tau_t, ref_t)
        suffix = fname.split('.tpr')[0]+"_"+str(i)
        shutil.move("new.tpr", f"{suffix}.tpr")
        run_cmd(f'gmx5 mdrun -deffnm {suffix} -v')


In [5]:
def MDP_Integer(fname, key, val):
    reader = TprReader(fname)
    reader.set_mdp_integer(key, val)

In [7]:
def get_xvf(fname, type):
    reader = TprReader(fname, bGRO=True)
    return reader.get_xvf(type)

In [9]:
if __name__ == '__main__':
    # arr = get_xvf('test/em.tpr', 'x')
    # print(arr)
    MD('test/em.tpr', 20)

TypeError: 'module' object is not callable